In [1]:
from pathlib import Path

import pandas as pd
from transformers import AutoTokenizer

PROJECT_ROOT = Path("/Users/shreybirmiwal/projects/gen-optimize-assembly")
CSV_PATH = PROJECT_ROOT / "src/codeforces-approach/data/codeforces_join.csv"
MODEL_NAME = "Qwen/Qwen2.5-Coder-0.5B"
MAX_SEQ_LENGTH = 2048
SPLITS_TO_CHECK = ["SFT", "VAL"]


def build_prompt(assembly_o0: str) -> str:
    return (
        "You are an expert compiler optimizer for Linux ARM64.\n"
        "Rewrite the given -O0 assembly into a more optimized version while preserving exact behavior.\n"
        "Return only assembly text for the same target toolchain.\n\n"
        "<INPUT_ASSEMBLY_O0>\n"
        f"{assembly_o0.strip()}\n"
        "</INPUT_ASSEMBLY_O0>\n\n"
        "<OPTIMIZED_ASSEMBLY>\n"
    )


tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("CSV:", CSV_PATH)
print("model:", MODEL_NAME)
print("max_seq_length:", MAX_SEQ_LENGTH)

/Users/shreybirmiwal/projects/gen-optimize-assembly/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


CSV: /Users/shreybirmiwal/projects/gen-optimize-assembly/src/codeforces-approach/data/codeforces_join.csv
model: Qwen/Qwen2.5-Coder-0.5B
max_seq_length: 2048


In [2]:
usecols = ["contest_id", "index", "dataset_split", "assembly_o0", "assembly_o3"]
df = pd.read_csv(CSV_PATH, usecols=usecols, keep_default_na=False)

df["dataset_split"] = df["dataset_split"].astype(str).str.upper()
df = df[df["dataset_split"].isin(SPLITS_TO_CHECK)].copy()
df = df[
    df["assembly_o0"].astype(str).str.strip().ne("")
    & df["assembly_o3"].astype(str).str.strip().ne("")
].reset_index(drop=True)

print("usable rows:", len(df))

records = []
for row in df.itertuples(index=False):
    prompt = build_prompt(row.assembly_o0)
    target = row.assembly_o3.strip() + tokenizer.eos_token

    prompt_ids = tokenizer(prompt, add_special_tokens=False, verbose=False)["input_ids"]
    target_ids = tokenizer(target, add_special_tokens=False, verbose=False)["input_ids"]
    full_ids = tokenizer(
        prompt + target,
        add_special_tokens=False,
        truncation=True,
        max_length=MAX_SEQ_LENGTH,
        verbose=False,
    )["input_ids"]

    prompt_token_count = min(len(prompt_ids), len(full_ids))
    target_tokens_after_trunc = len(full_ids) - prompt_token_count

    records.append(
        {
            "contest_id": row.contest_id,
            "index": row.index,
            "dataset_split": row.dataset_split,
            "prompt_tokens": len(prompt_ids),
            "target_tokens": len(target_ids),
            "tokens_after_trunc": len(full_ids),
            "target_tokens_after_trunc": target_tokens_after_trunc,
            "prompt_alone_exceeds_limit": len(prompt_ids) >= MAX_SEQ_LENGTH,
            "lost_all_target_tokens": target_tokens_after_trunc == 0,
            "lost_some_target_tokens": 0 < target_tokens_after_trunc < len(target_ids),
        }
    )

stats_df = pd.DataFrame(records)
stats_df.head()

usable rows: 5598


,contest_id,index,dataset_split,prompt_tokens,target_tokens,tokens_after_trunc,target_tokens_after_trunc,prompt_alone_exceeds_limit,lost_all_target_tokens,lost_some_target_tokens
0,852,A,SFT,13184,15700,2048,0,True,True,False
1,852,B,SFT,5807,6661,2048,0,True,True,False
2,852,C,SFT,42102,14047,2048,0,True,True,False
3,852,D,SFT,152218,31398,2048,0,True,True,False
4,852,E,SFT,3898,3119,2048,0,True,True,False


In [3]:
summary = pd.DataFrame(
    {
        "rows": [len(stats_df)],
        "prompt_alone_exceeds_limit": [int(stats_df["prompt_alone_exceeds_limit"].sum())],
        "lost_all_target_tokens": [int(stats_df["lost_all_target_tokens"].sum())],
        "lost_some_target_tokens": [int(stats_df["lost_some_target_tokens"].sum())],
        "median_prompt_tokens": [float(stats_df["prompt_tokens"].median())],
        "median_target_tokens": [float(stats_df["target_tokens"].median())],
        "median_target_tokens_after_trunc": [float(stats_df["target_tokens_after_trunc"].median())],
    }
)
summary

print("\nBy split")
print(
    stats_df.groupby("dataset_split")[[
        "prompt_alone_exceeds_limit",
        "lost_all_target_tokens",
        "lost_some_target_tokens",
    ]].mean().sort_index()
)

print("\nWorst rows (all target tokens lost)")
display(
    stats_df[stats_df["lost_all_target_tokens"]]
    .sort_values(["prompt_tokens", "target_tokens"], ascending=False)
    .head(20)
)

print("\nLargest prompts")
display(
    stats_df.sort_values("prompt_tokens", ascending=False).head(20)
)


By split
               prompt_alone_exceeds_limit  lost_all_target_tokens  \
dataset_split                                                       
SFT                              0.890996                0.890996   
VAL                              0.887500                0.887500   

               lost_some_target_tokens  
dataset_split                           
SFT                            0.08441  
VAL                            0.08500  

Worst rows (all target tokens lost)


,contest_id,index,dataset_split,prompt_tokens,target_tokens,tokens_after_trunc,target_tokens_after_trunc,prompt_alone_exceeds_limit,lost_all_target_tokens,lost_some_target_tokens
1251,1709,F,SFT,465583,146626,2048,0,True,True,False
1986,576,E,SFT,453058,157960,2048,0,True,True,False
638,1193,C,SFT,413177,100849,2048,0,True,True,False
4271,1707,E,SFT,399517,105338,2048,0,True,True,False
192,1648,E,SFT,392437,55529,2048,0,True,True,False
1594,1776,E,SFT,391832,64228,2048,0,True,True,False
4662,1732,D2,SFT,388891,46030,2048,0,True,True,False
1274,690,F2,SFT,372435,76020,2048,0,True,True,False
1452,843,E,SFT,368274,71927,2048,0,True,True,False
529,1383,F,SFT,339652,61159,2048,0,True,True,False



Largest prompts


,contest_id,index,dataset_split,prompt_tokens,target_tokens,tokens_after_trunc,target_tokens_after_trunc,prompt_alone_exceeds_limit,lost_all_target_tokens,lost_some_target_tokens
1251,1709,F,SFT,465583,146626,2048,0,True,True,False
1986,576,E,SFT,453058,157960,2048,0,True,True,False
638,1193,C,SFT,413177,100849,2048,0,True,True,False
4271,1707,E,SFT,399517,105338,2048,0,True,True,False
192,1648,E,SFT,392437,55529,2048,0,True,True,False
1594,1776,E,SFT,391832,64228,2048,0,True,True,False
4662,1732,D2,SFT,388891,46030,2048,0,True,True,False
1274,690,F2,SFT,372435,76020,2048,0,True,True,False
1452,843,E,SFT,368274,71927,2048,0,True,True,False
529,1383,F,SFT,339652,61159,2048,0,True,True,False


In [ ]:
// can we see the generation -- ie run a generation and see the output
// can we compare it agianst base model